In [96]:
import os
import re
import io
import csv
import duckdb
import numpy as np
import pandas as pd
from pathlib import Path
from unidecode import unidecode

path = os.getcwd()

In [97]:
def infer_skiprows(path, sep= ";", max_rows=30):
    match_header = re.compile(r"em r\$ mil", re.IGNORECASE)

    with open(path, encoding="utf-8-sig") as f:
        reader = csv.reader(f, delimiter=sep)
        rows = [r for _, r in zip(range(max_rows), reader)]
        
    matches = [[bool(match_header.search(c)) for c in r] for r in rows]
    n_matches = [sum(m) for m in matches]
    hits = [i for i, e in enumerate(n_matches) if e > 0]
    
    if not hits:
        return 0
    else:
        return min(hits) + 1

def normalize_headers(path, skiprows, num_headers):
    txt = Path(path).read_text(encoding="utf-8-sig")
    lines = txt.splitlines(keepends=True)

    start = skiprows
    end = skiprows + num_headers

    headers = lines[start:end]

    max_seps = max(h.count(";") + 1 for h in headers)

    for i, h in enumerate(headers):
        seps = h.count(";") + 1
        if seps < max_seps:
            seps_missing = max_seps - seps
            h = h.rstrip("\n") + ";" * seps_missing + "\n"
            headers[i] = h
            
    lines[start:end] = headers
    return "".join(lines)

def infer_headers(path, sep=";", max_rows=30, min_hits=1, skiprows=0):
    match_number = re.compile(r"\d+\.\d+")
    
    with open(path, encoding="utf-8-sig") as f:
        reader = csv.reader(f, delimiter=sep)
        
        for _ in range(skiprows):
            next(reader)
        
        rows = [r for _, r in zip(range(max_rows), reader)]
   
        
    matches = [[bool(match_number.search(c)) for c in r] for r in rows]
    n_matches = [sum(m) for m in matches]
    
    return min([i for i, e in enumerate(n_matches) if e >= min_hits])

In [98]:
# funcao que limpa nome de colunas
def clean_names(df, i, fill=None):
    clean = (
            pd.Series(df.columns.get_level_values(i), dtype = "string")
                .str.strip()
                .str.lower()
                .str.replace(" ", "_") # replaces spaces with _
                .str.replace(r"[^\w\s]", "", regex = True) # removes /, () and []
                .map(unidecode) # remove special characters
                .replace(r"^unnamed.*", "", regex=True)
    )
    
    if fill is None:
        return clean
    elif fill == "simple":
        return clean.replace("", pd.NA).ffill().fillna("")
    elif fill == "inside":
        return clean.replace("", pd.NA).ffill(limit_area = "inside").fillna("")
    
    raise ValueError(f"fill must be None, 'simple', or 'inside' (got {fill!r})")

# criando funcao pra ler arquivos
def read_ifdata(path, sep=";", skiprows=None, num_headers=1, normalize=False):
    if not normalize:
        txt = Path(path).read_text(encoding="utf-8-sig")
        txt = re.sub(r";(?=\r?\n)", "", txt)  # remove trailing ';' at end of each line
    else:
        txt = normalize_headers(path=path, skiprows=skiprows, num_headers=num_headers)
        
    headers = list(range(num_headers))
    
    df = pd.read_csv(io.StringIO(txt), sep=sep, skiprows=skiprows, header=headers)

    if num_headers > 1:
        cols = []
        for i in headers:
            if i == 0:
                s = clean_names(df, i, fill="simple")
            elif i == 1:
                s = clean_names(df, i, fill="inside")
            else:
                s = clean_names(df, i)      
            cols.append(s)

        flat = []
        for parts in zip(*cols): # parts is a tuple like (level0, level1, level2, ...)
            parts = [p for p in parts if p]  # drop empties: (level_1, "") -> (level_1)
            out = []
            for p in parts: # iters through each part inside a tuple
                if not out or out[-1] != p: # append the first part and dont append if the last part appended is equal to the next
                    out.append(p)
            flat.append("_".join(out)) # joins everything and becomes level_1_level_2_level_3
            
        df.columns = flat
        return df
    else:
        cols = clean_names(df, 0)
        df.columns = cols
        return df

In [104]:
con = duckdb.connect(path + "/ifdata.duckdb")

In [100]:
# itera pelos dataframes e faz a ingestao no duckdb
root = Path(path + "/dfs_by_report")
reports = set()

for file in root.rglob("*.csv"):
        print("Working on " + file.name)
        report = file.parent.parent.name 
        inst_type = file.parent.name
        
        skiprows = infer_skiprows(file)
        n_headers = infer_headers(file, skiprows=skiprows)
        
        if file.parent.name != "legado":
            df = read_ifdata(file, skiprows=skiprows, num_headers=n_headers)
        else:
            df = read_ifdata(file, skiprows=skiprows, num_headers=n_headers, normalize=True)

        df["tipo_if"] = inst_type
        df = df.astype("string")
        
        # evita colunas duplicadas
        s = df.columns.to_series().groupby(df.columns)
        n = s.cumcount()  # 0,1,2,...

        df.columns = np.where(
            n > 0,
            df.columns + "_" + (n + 1).astype(str),
            df.columns
        )
        
        if report not in reports:
            con.register("temp", df)
            con.execute(f"""
                CREATE TABLE IF NOT EXISTS {report} AS
                SELECT * FROM temp LIMIT 0
            """)
            con.execute(f"INSERT INTO {report} SELECT * FROM temp")
            con.unregister("temp")
            reports.add(report)
        else:
            existing = con.execute(f"SELECT * FROM {report} LIMIT 0").fetch_df().columns
            current = df.columns
            
            not_in_table = [c for c in current if c not in existing]
            not_in_df = [c for c in existing if c not in current]
            
            if not_in_table:
                for c in not_in_table:
                    con.execute(f"ALTER TABLE {report} ADD COLUMN {c} VARCHAR")
            if not_in_df:
                for c in not_in_df:
                    df[c] = pd.NA
                
            existing = con.execute(f"SELECT * FROM {report} LIMIT 0").fetch_df().columns
            df = df[existing]
            con.register("temp", df)
            con.execute(f"INSERT INTO {report} SELECT * FROM temp")
            con.unregister("temp")
                
con.close()

Working on 199512_Depósito.csv
Working on 199412_Depósito.csv
Working on 199812_Depósito.csv
Working on 199612_Depósito.csv
Working on 199606_Depósito.csv
Working on 199712_Depósito.csv
Working on 199806_Depósito.csv
Working on 199912_Depósito.csv
Working on 199506_Depósito.csv
Working on 199706_Depósito.csv
Working on 199906_Depósito.csv
Working on 032022_Instituições_com_Operações_de_Câmbio_Movimentação_de_Câmbio_no_Trimestre.csv
Working on 032021_Instituições_com_Operações_de_Câmbio_Movimentação_de_Câmbio_no_Trimestre.csv
Working on 122020_Instituições_com_Operações_de_Câmbio_Movimentação_de_Câmbio_no_Trimestre.csv
Working on 092020_Instituições_com_Operações_de_Câmbio_Movimentação_de_Câmbio_no_Trimestre.csv
Working on 062022_Instituições_com_Operações_de_Câmbio_Movimentação_de_Câmbio_no_Trimestre.csv
Working on 122016_Instituições_com_Operações_de_Câmbio_Movimentação_de_Câmbio_no_Trimestre.csv
Working on 122019_Instituições_com_Operações_de_Câmbio_Movimentação_de_Câmbio_no_Trimestr